In [ ]:
!pip install pycryptodome

In [ ]:
# Importamos la librería para cifrado AES y manejo de archivos
from Crypto.Cipher import AES
import os

# Función para añadir relleno (padding) PKCS7 al texto plano
def pad(data):
    # Calcula cuántos bytes de relleno se necesitan
    padding_length = 16 - len(data) % 16
    # Crea el relleno con el byte correspondiente al número de bytes que faltan
    padding = bytes([padding_length] * padding_length)
    # Retorna los datos originales con el relleno añadido
    return data + padding

# Función para eliminar el relleno del texto plano después del descifrado
def unpad(data):
    # El último byte contiene la longitud del relleno
    padding_length = data[-1]
    # Verifica que el relleno sea válido
    if padding_length < 1 or padding_length > 16:
        raise ValueError("Padding inválido detectado")
    # Retorna los datos originales sin el relleno
    return data[:-padding_length]


# Función para cifrar un archivo usando AES en modo CBC
def encrypt_file(input_file, output_file):
    # Genera una clave aleatoria de 256 bits (32 bytes)
    key = os.urandom(32)
    # Genera un Vector de Inicialización (IV) aleatorio de 128 bits (16 bytes)
    iv = os.urandom(16)
    # Inicializa el cifrador AES en modo CBC con la clave y el IV
    cipher = AES.new(key, AES.MODE_CBC, iv)
    try:
        # Lee el archivo original (plaintext)
        with open(input_file, 'rb') as f:
            plaintext = f.read()
        # Añade relleno al plaintext para asegurar que es múltiplo de 16
        padded_plaintext = pad(plaintext)
        # Realiza el cifrado del plaintext con relleno
        ciphertext = cipher.encrypt(padded_plaintext)
        # Guarda el IV y ciphertext juntos en el archivo cifrado
        with open(output_file, 'wb') as f:
            f.write(iv + ciphertext)  # IV se guarda al inicio
        # Devuelve la clave e IV como cadenas hexadecimales para su almacenamiento seguro
        return key.hex(), iv.hex()
    except Exception as e:
        print(f"Error durante el cifrado: {e}")
        return None, None

# Función para descifrar un archivo cifrado con AES en modo CBC
def decrypt_file(input_file, output_file, key_hex):
    # Convierte la clave de formato hexadecimal a bytes
    key = bytes.fromhex(key_hex)
    try:
        # Lee el archivo cifrado
        with open(input_file, 'rb') as f:
            iv = f.read(16)  # Extrae los primeros 16 bytes como IV
            ciphertext = f.read()  # El resto es el texto cifrado
        # Inicializa el descifrador AES en modo CBC con la clave y IV
        cipher = AES.new(key, AES.MODE_CBC, iv)
        # Realiza el descifrado del ciphertext
        padded_plaintext = cipher.decrypt(ciphertext)
        # Elimina el relleno para obtener el plaintext original
        plaintext = unpad(padded_plaintext)
        # Guarda el plaintext en el archivo descifrado
        with open(output_file, 'wb') as f:
            f.write(plaintext)
    except Exception as e:
        print(f"Error durante el descifrado: {e}")

In [ ]:

    while True:
        print("\n--- MENÚ ---")
        print("1. Encriptar")
        print("2. Desencriptar")
        print("3. Salir")

        opcion = input("Elige (1, 2 o 3): ")

        match opcion:

            case "1":
                archivo = input("Archivo a encriptar: ")
                if os.path.exists(archivo):
                    clave, iv = encrypt_file(archivo, "Cifrado.txt")
                    print(f"Encriptado en 'Cifrado.txt'.\n GUARDA ESTA CLAVE: {clave}")
                else:
                    print("Archivo no encontrado.")

            case "2":
                archivo = input("Archivo a desencriptar: ")
                if os.path.exists(archivo):
                    clave = input("Pega tu clave: ")
                    decrypt_file(archivo, "Desencriptado.txt", clave)
                    print("¡Desencriptado en 'Desencriptado.txt'!")
                else:
                    print("Archivo no encontrado.")

            case "3":
                print("Saliendo...")
                break


            case _:
                print("Opción incorrecta.")